# Tracing and Evaluating Hawkish-Dovish Models for FOMC Text

This tutorial demonstrates how to use W&B Weave to trace,
evaluate, and compare machine learning models that classify
Federal Reserve communications as hawkish, dovish, or neutral.

We also investigate the relationship between FOMC statement
tone and changes in the 2-year Treasury yield.

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

# System libraries.
import logging

# Third-party libraries.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# # To install additional packages, use:
# import helpers.hmodule as hmodule
# hmodule.install_module_if_not_present(
#     ["pycaret"],
#     use_activate=True,
#     use_sudo=False,
#     venv_path="/opt/venv",
# )

In [3]:
# Use this for most notebooks.
import helpers.hdbg as hdbg
import helpers.hnotebook as hnotebook

_LOG = logging.getLogger(__name__)

# Initialize notebook configuration and logging.
hdbg.init_logger(verbosity=logging.INFO)
hnotebook.config_notebook()

# Convert `display` into `print()`.
try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore

INFO  > /usr/local/lib/python3.12/site-packages/ipykernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-bfa00023-d401-4875-91f1-24a15d9b31fa.json


In [4]:
from pathlib import Path
import pandas as pd
import requests

# Jupyter notebook's current working directory.
PROJECT_DIR = Path.cwd()

# Directories for caching downloaded data.
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project directory: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1
Raw data directory: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1/data/raw
Processed data directory: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1/data/processed


## Data Ingestion and Caching Labeled FOMC Hawkish/Dovish Sentences from Hugging Face

In [5]:
BASE_URL = (
    "https://huggingface.co/datasets/"
    "gtfintechlab/fomc_communication/resolve/main"
)

def download_csv(filename):
    destination = RAW_DIR / filename

    if destination.exists():
        print(f"Using cached file: {filename}")
    else:
        url = f"{BASE_URL}/{filename}"
        print(f"Downloading {filename}...")

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        # Write the downloaded file to the cache.
        destination.write_bytes(response.content)
        print(f"Saved: {destination}")

    return pd.read_csv(destination)


train_df = download_csv("train.csv")
test_df = download_csv("test.csv")

print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

Saved: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1/data/raw/train.csv
Saved: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1/data/raw/test.csv
Training data shape: (1984, 5)
Testing data shape: (496, 5)


In [6]:
print("TRAINING DATA")
display(train_df.head())

print("\nTESTING DATA")
display(test_df.head())

print("\nCOLUMN NAMES")
print(train_df.columns.tolist())

print("\nMISSING VALUES")
print(train_df.isnull().sum())

TRAINING DATA


,index,sentence,year,label,orig_index
0,122,Broad equity price indexes fell sharply over t...,2011,2,119
1,275,"For example, although a number of features of ...",2004,2,270
2,632,"On balance, most participants still expected i...",2005,2,599
3,486,Low readings on overall and core consumer pric...,2008,0,465
4,74,But I want to emphasize that we do have a comm...,2011,0,73



TESTING DATA


,index,sentence,year,label,orig_index
0,659,Participants agreed that the labor market had ...,2004,2,625
1,318,"You know, stimulating aggregate demand is one ...",2014,0,311
2,584,Much of this performance was fueled by an inve...,2013,0,566
3,1003,With about $250 billion of these inflation-pro...,2000,2,974
4,17,A few participants judged that while the labor...,2020,2,17



COLUMN NAMES
['index', 'sentence', 'year', 'label', 'orig_index']

MISSING VALUES
index         0
sentence      0
year          0
label         0
orig_index    0
dtype: int64


In [7]:
# Check the number of sentences in each dataset.
print("Training sentences:", len(train_df))
print("Testing sentences:", len(test_df))

# Check the available labels.
print("\nTraining label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nTesting label distribution:")
print(test_df["label"].value_counts().sort_index())

# Inspect sample sentences for each label.
for label in sorted(train_df["label"].unique()):
    print(f"\n--- Label {label} ---")
    samples = train_df[
        train_df["label"] == label
    ]["sentence"].head(3)

    for sentence in samples:
        print("-", sentence)

Training sentences: 1984
Testing sentences: 496

Training label distribution:
label
0    515
1    492
2    977
Name: count, dtype: int64

Testing label distribution:
label
0    135
1    114
2    247
Name: count, dtype: int64

--- Label 0 ---
- Low readings on overall and core consumer price inflation in recent months, as well as the weakened economic outlook, kept near-term inflation expectations reported in surveys well below their high levels in mid-2008.
- But I want to emphasize that we do have a commitment to raising inflation to 2 percent.
- Nonetheless, participants viewed the housing situation and its potential further effect on employment, income, and wealth as one of the major sources of downside risk to the economic outlook.

--- Label 1 ---
- Our new statement explicitly acknowledges the challenges posed by the proximity of interest rates to the effective lower bound.
- Many observed, however, that a favorable outcome to the hostilities in the Middle East and lower oil pric

In [8]:
# Preserve the original datasets.
train_clean = train_df.copy()
test_clean = test_df.copy()

# Keep the relevant columns.
columns = ["sentence", "year", "label"]

train_clean = train_clean[columns]
test_clean = test_clean[columns]

# Remove leading/trailing whitespace.
train_clean["sentence"] = (
    train_clean["sentence"].astype(str).str.strip()
)

test_clean["sentence"] = (
    test_clean["sentence"].astype(str).str.strip()
)

# Check for empty sentences.
assert train_clean["sentence"].ne("").all()
assert test_clean["sentence"].ne("").all()

# Verify the labels are valid numeric classes.
assert set(train_clean["label"].unique()).issubset({0, 1, 2})
assert set(test_clean["label"].unique()).issubset({0, 1, 2})

# Cache the processed datasets.
train_clean.to_csv(
    PROCESSED_DIR / "train_clean.csv",
    index=False
)

test_clean.to_csv(
    PROCESSED_DIR / "test_clean.csv",
    index=False
)

print("Processed datasets saved successfully!")
print("Training shape:", train_clean.shape)
print("Testing shape:", test_clean.shape)

Processed datasets saved successfully!
Training shape: (1984, 3)
Testing shape: (496, 3)


In [9]:
print("Processed data files:")
for file in PROCESSED_DIR.glob("*.csv"):
    print(file.name, "-", file.stat().st_size, "bytes")

print("\nCleaned training data:")
display(train_clean.head())

print("\nCleaned testing data:")
display(test_clean.head())

Processed data files:
test_clean.csv - 100037 bytes
train_clean.csv - 407011 bytes

Cleaned training data:


,sentence,year,label
0,Broad equity price indexes fell sharply over t...,2011,2
1,"For example, although a number of features of ...",2004,2
2,"On balance, most participants still expected i...",2005,2
3,Low readings on overall and core consumer pric...,2008,0
4,But I want to emphasize that we do have a comm...,2011,0



Cleaned testing data:


,sentence,year,label
0,Participants agreed that the labor market had ...,2004,2
1,"You know, stimulating aggregate demand is one ...",2014,0
2,Much of this performance was fueled by an inve...,2013,0
3,With about $250 billion of these inflation-pro...,2000,2
4,A few participants judged that while the labor...,2020,2


In [10]:
# Verify the processed files exist.
train_path = PROCESSED_DIR / "train_clean.csv"
test_path = PROCESSED_DIR / "test_clean.csv"

assert train_path.exists(), "Training file missing"
assert test_path.exists(), "Testing file missing"

# Reload the cached datasets.
train_cached = pd.read_csv(train_path)
test_cached = pd.read_csv(test_path)

# Confirm the saved data matches the originals.
assert len(train_cached) == 1984
assert len(test_cached) == 496

assert train_cached["sentence"].notna().all()
assert test_cached["sentence"].notna().all()

print("Data successfully cleaned and cached!")
print("Training shape:", train_cached.shape)
print("Testing shape:", test_cached.shape)

Data successfully cleaned and cached!
Training shape: (1984, 3)
Testing shape: (496, 3)


## FOMC Statement Data Ingestion (2015–2024)

This section downloads and caches official Federal Open Market
Committee monetary policy statements from the Federal Reserve.

The statements will be used to calculate hawkish-dovish tone
scores and analyze their relationship with changes in the
2-year Treasury yield.

In [11]:
from urllib.parse import urljoin
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_FED_URL = "https://www.federalreserve.gov"

calendar_urls = [
    f"{BASE_FED_URL}/monetarypolicy/fomchistorical{year}.htm"
    for year in range(2015, 2021)
]

calendar_urls.append(
    f"{BASE_FED_URL}/monetarypolicy/fomccalendars.htm"
)

# Match official monetary policy statement URLs.
statement_pattern = re.compile(
    r"/newsevents/pressreleases/monetary(\d{8})a\.htm",
    re.IGNORECASE
)

statement_links = {}

for calendar_url in calendar_urls:
    response = requests.get(calendar_url, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for link in soup.find_all("a", href=True):
        full_url = urljoin(calendar_url, link["href"])
        match = statement_pattern.search(full_url)

        if match:
            date = pd.to_datetime(
                match.group(1),
                format="%Y%m%d"
            )

            if 2015 <= date.year <= 2024:
                statement_links[date.strftime("%Y-%m-%d")] = full_url

# Create a table of discovered statements.
fomc_links_df = pd.DataFrame(
    [
        {"date": date, "source_url": url}
        for date, url in statement_links.items()
    ]
)

fomc_links_df = fomc_links_df.sort_values(
    "date"
).reset_index(drop=True)

print("Statements discovered:", len(fomc_links_df))

print("\nStatements by year:")
if not fomc_links_df.empty:
    print(
        pd.to_datetime(fomc_links_df["date"])
        .dt.year.value_counts()
        .sort_index()
    )

display(fomc_links_df.head(10))
display(fomc_links_df.tail(10))

Statements discovered: 85

Statements by year:
date
2015     8
2016     8
2017     8
2018     8
2019     9
2020    12
2021     8
2022     8
2023     8
2024     8
Name: count, dtype: int64


,date,source_url
0,2015-01-28,https://www.federalreserve.gov/newsevents/pres...
1,2015-03-18,https://www.federalreserve.gov/newsevents/pres...
2,2015-04-29,https://www.federalreserve.gov/newsevents/pres...
3,2015-06-17,https://www.federalreserve.gov/newsevents/pres...
4,2015-07-29,https://www.federalreserve.gov/newsevents/pres...
5,2015-09-17,https://www.federalreserve.gov/newsevents/pres...
6,2015-10-28,https://www.federalreserve.gov/newsevents/pres...
7,2015-12-16,https://www.federalreserve.gov/newsevents/pres...
8,2016-01-27,https://www.federalreserve.gov/newsevents/pres...
9,2016-03-16,https://www.federalreserve.gov/newsevents/pres...


,date,source_url
75,2023-11-01,https://www.federalreserve.gov/newsevents/pres...
76,2023-12-13,https://www.federalreserve.gov/newsevents/pres...
77,2024-01-31,https://www.federalreserve.gov/newsevents/pres...
78,2024-03-20,https://www.federalreserve.gov/newsevents/pres...
79,2024-05-01,https://www.federalreserve.gov/newsevents/pres...
80,2024-06-12,https://www.federalreserve.gov/newsevents/pres...
81,2024-07-31,https://www.federalreserve.gov/newsevents/pres...
82,2024-09-18,https://www.federalreserve.gov/newsevents/pres...
83,2024-11-07,https://www.federalreserve.gov/newsevents/pres...
84,2024-12-18,https://www.federalreserve.gov/newsevents/pres...


In [12]:
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

# Directory for cached FOMC statements.
FOMC_DIR = RAW_DIR / "fomc_statements"
FOMC_DIR.mkdir(parents=True, exist_ok=True)

def clean_fomc_statement(text):
    """Normalize whitespace and clean extracted FOMC statement text."""
    if not isinstance(text, str):
        raise TypeError("FOMC statement text must be a string.")

    # Normalize non-breaking spaces and common whitespace artifacts.
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def download_fomc_statement(date, url):
    """Download, clean, and cache an FOMC statement."""

    cache_path = FOMC_DIR / f"{date}.txt"

    # Reuse cached files when available.
    if cache_path.exists():
        text = cache_path.read_text(encoding="utf-8")
        if text.strip():
            return text

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    article = soup.select_one("#article")

    if article is None:
        raise ValueError(f"No article found for {date}")

    paragraphs = [
        p.get_text(" ", strip=True)
        for p in article.find_all("p")
    ]

    raw_text = "\n".join(
        p for p in paragraphs if p
    )

    cleaned_text = clean_fomc_statement(raw_text)

    if len(cleaned_text) < 300:
        raise ValueError(
            f"Statement appears too short: {date}"
        )

    cache_path.write_text(
        cleaned_text,
        encoding="utf-8"
    )

    return cleaned_text


records = []
failed_downloads = []

for row in fomc_links_df.itertuples(index=False):
    try:
        text = download_fomc_statement(
            row.date,
            row.source_url
        )

        records.append({
            "date": row.date,
            "statement_text": text,
            "source_url": row.source_url
        })

        print(f"Loaded: {row.date}")

        # Avoid sending requests too quickly.
        time.sleep(0.3)

    except Exception as error:
        failed_downloads.append({
            "date": row.date,
            "error": str(error)
        })
        print(f"FAILED: {row.date} - {error}")


fomc_df = pd.DataFrame(
    records,
    columns=["date", "statement_text", "source_url"]
)

print("\nDownload complete!")
print("Statements loaded:", len(fomc_df))
print("Failed downloads:", len(failed_downloads))

Loaded: 2015-01-28
Loaded: 2015-03-18
Loaded: 2015-04-29
Loaded: 2015-06-17
Loaded: 2015-07-29
Loaded: 2015-09-17
Loaded: 2015-10-28
Loaded: 2015-12-16
Loaded: 2016-01-27
Loaded: 2016-03-16
Loaded: 2016-04-27
Loaded: 2016-06-15
Loaded: 2016-07-27
Loaded: 2016-09-21
Loaded: 2016-11-02
Loaded: 2016-12-14
Loaded: 2017-02-01
Loaded: 2017-03-15
Loaded: 2017-05-03
Loaded: 2017-06-14
Loaded: 2017-07-26
Loaded: 2017-09-20
Loaded: 2017-11-01
Loaded: 2017-12-13
Loaded: 2018-01-31
Loaded: 2018-03-21
Loaded: 2018-05-02
Loaded: 2018-06-13
Loaded: 2018-08-01
Loaded: 2018-09-26
Loaded: 2018-11-08
Loaded: 2018-12-19
Loaded: 2019-01-30
Loaded: 2019-03-20
Loaded: 2019-05-01
Loaded: 2019-06-19
Loaded: 2019-07-31
Loaded: 2019-09-18
Loaded: 2019-10-11
Loaded: 2019-10-30
Loaded: 2019-12-11
Loaded: 2020-01-29
Loaded: 2020-03-03
Loaded: 2020-03-15
Loaded: 2020-03-23
Loaded: 2020-03-31
Loaded: 2020-04-29
Loaded: 2020-06-10
Loaded: 2020-07-29
Loaded: 2020-08-27
Loaded: 2020-09-16
Loaded: 2020-11-05
Loaded: 2020

In [13]:
# Check statement counts by year.
fomc_df["date"] = pd.to_datetime(fomc_df["date"])

print("Statements by year:")
print(
    fomc_df["date"]
    .dt.year.value_counts()
    .sort_index()
)

# Check missing values and duplicate dates.
print("\nMissing values:")
print(fomc_df.isnull().sum())

print("\nDuplicate dates:")
print(fomc_df["date"].duplicated().sum())

# Inspect statement lengths to identify suspicious records.
fomc_df["text_length"] = (
    fomc_df["statement_text"].str.len()
)

print("\nShortest statements:")
display(
    fomc_df.nsmallest(
        10, "text_length"
    )[["date", "text_length", "statement_text"]]
)

print("\nFailed downloads:")
display(pd.DataFrame(failed_downloads))

Statements by year:
date
2015     8
2016     8
2017     8
2018     8
2019     9
2020    12
2021     8
2022     8
2023     8
2024     8
Name: count, dtype: int64

Missing values:
date              0
statement_text    0
source_url        0
dtype: int64

Duplicate dates:
0

Shortest statements:


,date,text_length,statement_text
42,2020-03-03,915,"March 03, 2020\nFor release at 10:00 a.m. EST ..."
45,2020-03-31,1704,"March 31, 2020\nFor release at 8:30 a.m. EDT S..."
29,2018-09-26,2027,"September 26, 2018\nFor release at 2:00 p.m. E..."
30,2018-11-08,2102,"November 08, 2018\nFor release at 2:00 p.m. ES..."
28,2018-08-01,2151,"August 01, 2018\nFor release at 2:00 p.m. EDT ..."
65,2022-07-27,2164,"July 27, 2022\nFor release at 2:00 p.m. EDT Sh..."
66,2022-09-21,2166,"September 21, 2022\nFor release at 2:00 p.m. E..."
40,2019-12-11,2203,"December 11, 2019\nFor release at 2:00 p.m. ES..."
34,2019-05-01,2207,"May 01, 2019\nFor release at 2:00 p.m. EDT Sha..."
41,2020-01-29,2210,"January 29, 2020\nFor release at 2:00 p.m. EST..."



Failed downloads:


""


In [14]:
# Identify statements that may not represent FOMC policy decisions.

policy_phrases = [
    "federal open market committee",
    "the committee decided",
    "the committee seeks",
    "target range for the federal funds rate",
    "target for the federal funds rate",
]

def contains_policy_language(text):
    text = text.lower()
    return any(phrase in text for phrase in policy_phrases)

fomc_df["has_policy_language"] = (
    fomc_df["statement_text"]
    .apply(contains_policy_language)
)

review_df = fomc_df[
    ~fomc_df["has_policy_language"]
].copy()

print("Announcements requiring review:", len(review_df))

display(
    review_df[
        ["date", "text_length", "source_url"]
    ]
)

Announcements requiring review: 1


,date,text_length,source_url
45,2020-03-31,1704,https://www.federalreserve.gov/newsevents/pres...


In [15]:
def fix_text_encoding(text):
    """Repair common UTF-8 text decoding artifacts."""
    if not isinstance(text, str):
        return text

    if "â" in text or "Ã" in text:
        try:
            text = text.encode("latin-1").decode("utf-8")
        except UnicodeError:
            pass

    return text


fomc_df["statement_text"] = (
    fomc_df["statement_text"]
    .apply(fix_text_encoding)
)

# Check the March 3 statement.
march3 = fomc_df.loc[
    fomc_df["date"] == pd.Timestamp("2020-03-03"),
    "statement_text"
].iloc[0]

print(march3)

March 03, 2020
For release at 10:00 a.m. EST Share
The fundamentals of the U.S. economy remain strong. However, the coronavirus poses evolving risks to economic activity. In light of these risks and in support of achieving its maximum employment and price stability goals, the Federal Open Market Committee decided today to lower the target range for the federal funds rate by 1/2 percentage point, to 1 to 1‑1/4 percent. The Committee is closely monitoring developments and their implications for the economic outlook and will use its tools and act as appropriate to support the economy.
Voting for the monetary policy action were Jerome H. Powell, Chair; John C. Williams, Vice Chair; Michelle W. Bowman; Lael Brainard; Richard H. Clarida; Patrick Harker; Robert S. Kaplan; Neel Kashkari; Loretta J. Mester; and Randal K. Quarles.
For media inquiries, call 202-452-2955.
Implementation Note issued March 3, 2020


In [16]:
# Keep the original dataframe for auditing.
fomc_all_df = fomc_df.copy()

# Exclude the unrelated FIMA Repo Facility announcement.
excluded_dates = [
    pd.Timestamp("2020-03-31")
]

fomc_filtered_df = fomc_df[
    ~fomc_df["date"].isin(excluded_dates)
].copy()

fomc_filtered_df = fomc_filtered_df.sort_values(
    "date"
).reset_index(drop=True)

print("Original announcements:", len(fomc_df))
print("Remaining announcements:", len(fomc_filtered_df))
print("Excluded announcements:", len(fomc_df) - len(fomc_filtered_df))

Original announcements: 85
Remaining announcements: 84
Excluded announcements: 1


In [17]:
# Prepare the combined FOMC dataset.
fomc_final_df = fomc_filtered_df[
    ["date", "statement_text", "source_url"]
].copy()

fomc_final_df = fomc_final_df.sort_values(
    "date"
).reset_index(drop=True)

# Verify basic dataset integrity.
assert len(fomc_final_df) == 84
assert fomc_final_df["date"].is_unique
assert fomc_final_df["statement_text"].notna().all()
assert fomc_final_df["statement_text"].str.strip().ne("").all()

# Save to the processed data directory.
fomc_output_path = (
    PROCESSED_DIR / "fomc_statements_2015_2024.csv"
)

fomc_final_df.to_csv(
    fomc_output_path,
    index=False,
    encoding="utf-8"
)

print("FOMC dataset saved successfully!")
print("File:", fomc_output_path)
print("Statements:", len(fomc_final_df))

display(fomc_final_df.head())

FOMC dataset saved successfully!
File: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1/data/processed/fomc_statements_2015_2024.csv
Statements: 84


,date,statement_text,source_url
0,2015-01-28,"January 28, 2015\nFor immediate release Share\...",https://www.federalreserve.gov/newsevents/pres...
1,2015-03-18,"March 18, 2015\nFor immediate release Share\nI...",https://www.federalreserve.gov/newsevents/pres...
2,2015-04-29,"April 29, 2015\nFor immediate release Share\nI...",https://www.federalreserve.gov/newsevents/pres...
3,2015-06-17,"June 17, 2015\nFor immediate release Share\nIn...",https://www.federalreserve.gov/newsevents/pres...
4,2015-07-29,"July 29, 2015\nFor immediate release Share\nIn...",https://www.federalreserve.gov/newsevents/pres...


In [18]:
# Reload the saved CSV to verify it.
fomc_cached_df = pd.read_csv(
    fomc_output_path,
    parse_dates=["date"]
)

assert len(fomc_cached_df) == 84
assert fomc_cached_df["date"].is_unique
assert fomc_cached_df["statement_text"].notna().all()

print("FOMC dataset verified successfully!")
print("Shape:", fomc_cached_df.shape)

print("\nStatements by year:")
print(
    fomc_cached_df["date"]
    .dt.year.value_counts()
    .sort_index()
)

FOMC dataset verified successfully!
Shape: (84, 3)

Statements by year:
date
2015     8
2016     8
2017     8
2018     8
2019     9
2020    11
2021     8
2022     8
2023     8
2024     8
Name: count, dtype: int64


## Daily 2-Year Treasury Yield Ingestion

Download and cache daily 2-year Treasury par yields from the
U.S. Department of the Treasury for 2014–2024.

In [19]:
from io import StringIO
from pathlib import Path
import time
import pandas as pd
import requests

TREASURY_DIR = RAW_DIR / "treasury_yields"
TREASURY_DIR.mkdir(parents=True, exist_ok=True)


def download_treasury_year(year, max_retries=5):
    cache_path = TREASURY_DIR / f"treasury_{year}.csv"

    # Reuse previously downloaded files.
    if cache_path.exists():
        print(f"Using cached Treasury data: {year}")
        return pd.read_csv(cache_path)

    url = (
        "https://home.treasury.gov/resource-center/"
        "data-chart-center/interest-rates/"
        f"daily-treasury-rates.csv/{year}/all"
    )

    params = {
        "type": "daily_treasury_yield_curve",
        "field_tdr_date_value": str(year),
        "page": "",
        "_format": "csv"
    }

    for attempt in range(1, max_retries + 1):
        try:
            print(
                f"Downloading {year} "
                f"(attempt {attempt}/{max_retries})..."
            )

            response = requests.get(
                url,
                params=params,
                timeout=90,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            response.raise_for_status()

            data = pd.read_csv(StringIO(response.text))
            data.columns = data.columns.str.strip()

            if "Date" not in data.columns or "2 Yr" not in data.columns:
                raise ValueError(
                    f"Unexpected columns for {year}: "
                    f"{data.columns.tolist()}"
                )

            if data.empty:
                raise ValueError(f"No data returned for {year}")

            # Cache only validated data.
            cache_path.write_text(
                response.text,
                encoding="utf-8"
            )

            print(f"Successfully downloaded {year}")
            return data

        except requests.exceptions.RequestException as error:
            print(f"Connection error: {error}")

            if attempt == max_retries:
                raise

            wait_time = min(5 * (2 ** (attempt - 1)), 60)
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)


# Download all years, using cached files when available.
treasury_frames = []

for year in range(2014, 2025):
    yearly_df = download_treasury_year(year)
    treasury_frames.append(yearly_df)

    # Brief pause between requests.
    time.sleep(1)

treasury_raw_df = pd.concat(
    treasury_frames,
    ignore_index=True
)

print("\nAll Treasury data downloaded!")
print("Total rows:", len(treasury_raw_df))
print("Columns:", treasury_raw_df.columns.tolist())

display(treasury_raw_df.head())

Successfully downloaded 2014
Successfully downloaded 2015
Successfully downloaded 2016
Successfully downloaded 2017
Successfully downloaded 2018
Successfully downloaded 2019
Successfully downloaded 2020
Successfully downloaded 2021
Successfully downloaded 2022
Successfully downloaded 2023
Successfully downloaded 2024

All Treasury data downloaded!
Total rows: 2751
Columns: ['Date', '1 Mo', '3 Mo', '6 Mo', '1 Yr', '2 Yr', '3 Yr', '5 Yr', '7 Yr', '10 Yr', '20 Yr', '30 Yr', '2 Mo', '4 Mo']


,Date,1 Mo,3 Mo,6 Mo,1 Yr,2 Yr,3 Yr,5 Yr,7 Yr,10 Yr,20 Yr,30 Yr,2 Mo,4 Mo
0,12/31/2014,0.03,0.04,0.12,0.25,0.67,1.10,1.65,1.97,2.17,2.47,2.75,NaN,NaN
1,12/30/2014,0.03,0.03,0.12,0.23,0.69,1.11,1.68,2.00,2.20,2.49,2.76,NaN,NaN
2,12/29/2014,0.01,0.03,0.12,0.25,0.72,1.14,1.72,2.02,2.22,2.51,2.78,NaN,NaN
3,12/26/2014,0.01,0.01,0.10,0.26,0.73,1.19,1.75,2.07,2.25,2.54,2.81,NaN,NaN
4,12/24/2014,0.01,0.01,0.14,0.26,0.73,1.18,1.76,2.09,2.27,2.56,2.83,NaN,NaN


In [20]:
# Standardize column names.
treasury_raw_df.columns = (
    treasury_raw_df.columns.str.strip()
)

# Keep only the date and 2-year Treasury yield.
treasury_df = treasury_raw_df[
    ["Date", "2 Yr"]
].copy()

treasury_df = treasury_df.rename(
    columns={
        "Date": "date",
        "2 Yr": "yield_2yr"
    }
)

# Convert dates and yields.
treasury_df["date"] = pd.to_datetime(
    treasury_df["date"],
    errors="raise"
)

treasury_df["yield_2yr"] = pd.to_numeric(
    treasury_df["yield_2yr"],
    errors="coerce"
)

# Remove rows with missing values.
treasury_df = treasury_df.dropna(
    subset=["date", "yield_2yr"]
)

# Sort by date.
treasury_df = treasury_df.sort_values(
    "date"
).reset_index(drop=True)

print("Treasury dataset shape:", treasury_df.shape)
print("Earliest date:", treasury_df["date"].min())
print("Latest date:", treasury_df["date"].max())

display(treasury_df.head())
display(treasury_df.tail())

Treasury dataset shape: (2751, 2)
Earliest date: 2014-01-02 00:00:00
Latest date: 2024-12-31 00:00:00


,date,yield_2yr
0,2014-01-02,0.39
1,2014-01-03,0.41
2,2014-01-06,0.40
3,2014-01-07,0.40
4,2014-01-08,0.43


,date,yield_2yr
2746,2024-12-24,4.29
2747,2024-12-26,4.30
2748,2024-12-27,4.31
2749,2024-12-30,4.24
2750,2024-12-31,4.25


In [21]:
print("Missing values:")
print(treasury_df.isnull().sum())

print("\nDuplicate dates:")
print(treasury_df["date"].duplicated().sum())

print("\nObservations by year:")
print(
    treasury_df["date"]
    .dt.year.value_counts()
    .sort_index()
)

print("\n2-year Treasury yield summary:")
print(treasury_df["yield_2yr"].describe())

# Validate data integrity.
assert treasury_df["date"].is_unique
assert treasury_df["yield_2yr"].notna().all()
assert treasury_df["date"].dt.year.between(
    2014, 2024
).all()

print("\nTreasury yield validation successful!")

Missing values:
date         0
yield_2yr    0
dtype: int64

Duplicate dates:
0

Observations by year:
date
2014    250
2015    251
2016    250
2017    250
2018    249
2019    250
2020    251
2021    251
2022    249
2023    250
2024    250
Name: count, dtype: int64

2-year Treasury yield summary:
count    2751.000000
mean        1.859055
std         1.562132
min         0.090000
25%         0.560000
50%         1.350000
75%         2.810000
max         5.190000
Name: yield_2yr, dtype: float64

Treasury yield validation successful!


In [22]:
treasury_output_path = (
    PROCESSED_DIR / "treasury_2yr_daily_2014_2024.csv"
)

treasury_df.to_csv(
    treasury_output_path,
    index=False,
    encoding="utf-8"
)

print("Treasury yield dataset saved successfully!")
print("File:", treasury_output_path)
print("Rows:", len(treasury_df))

Treasury yield dataset saved successfully!
File: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1/data/processed/treasury_2yr_daily_2014_2024.csv
Rows: 2751


## FOMC Statement and Treasury Yield Alignment

Align FOMC announcements with daily 2-year Treasury yields
and calculate changes in basis points relative to the
previous trading day's observation.

In [23]:
# Work with a copy of the Treasury data.
treasury_aligned = treasury_df.copy()

treasury_aligned = treasury_aligned.sort_values(
    "date"
).reset_index(drop=True)

# Previous available Treasury observation.
treasury_aligned["previous_trading_date"] = (
    treasury_aligned["date"].shift(1)
)

treasury_aligned["yield_previous_close"] = (
    treasury_aligned["yield_2yr"].shift(1)
)

# Yield on the current observation date.
treasury_aligned["yield_event_close"] = (
    treasury_aligned["yield_2yr"]
)

# Convert percentage-point changes to basis points.
treasury_aligned["yield_change_bps"] = (
    (
        treasury_aligned["yield_event_close"]
        - treasury_aligned["yield_previous_close"]
    ) * 100
).round(2)

display(treasury_aligned.head(10))

,date,yield_2yr,previous_trading_date,yield_previous_close,yield_event_close,yield_change_bps
0,2014-01-02,0.39,NaT,NaN,0.39,NaN
1,2014-01-03,0.41,2014-01-02,0.39,0.41,2.0
2,2014-01-06,0.40,2014-01-03,0.41,0.40,-1.0
3,2014-01-07,0.40,2014-01-06,0.40,0.40,0.0
4,2014-01-08,0.43,2014-01-07,0.40,0.43,3.0
5,2014-01-09,0.44,2014-01-08,0.43,0.44,1.0
6,2014-01-10,0.39,2014-01-09,0.44,0.39,-5.0
7,2014-01-13,0.39,2014-01-10,0.39,0.39,0.0
8,2014-01-14,0.39,2014-01-13,0.39,0.39,0.0
9,2014-01-15,0.41,2014-01-14,0.39,0.41,2.0


In [24]:
# Ensure both datasets use datetime dates.
fomc_final_df["date"] = pd.to_datetime(
    fomc_final_df["date"]
).dt.normalize()

treasury_aligned["date"] = pd.to_datetime(
    treasury_aligned["date"]
).dt.normalize()

# Keep the columns needed for alignment.
yield_columns = [
    "date",
    "previous_trading_date",
    "yield_previous_close",
    "yield_event_close",
    "yield_change_bps"
]

# Match FOMC announcement dates to Treasury dates.
fomc_yield_df = fomc_final_df.merge(
    treasury_aligned[yield_columns],
    on="date",
    how="left",
    validate="one_to_one"
)

print("Total FOMC announcements:", len(fomc_yield_df))

print(
    "Announcements with same-day yield data:",
    fomc_yield_df["yield_event_close"].notna().sum()
)

print(
    "Announcements missing same-day yield data:",
    fomc_yield_df["yield_event_close"].isna().sum()
)

display(fomc_yield_df.head())

Total FOMC announcements: 84
Announcements with same-day yield data: 83
Announcements missing same-day yield data: 1


,date,statement_text,source_url,previous_trading_date,yield_previous_close,yield_event_close,yield_change_bps
0,2015-01-28,"January 28, 2015\nFor immediate release Share\...",https://www.federalreserve.gov/newsevents/pres...,2015-01-27,0.54,0.50,-4.0
1,2015-03-18,"March 18, 2015\nFor immediate release Share\nI...",https://www.federalreserve.gov/newsevents/pres...,2015-03-17,0.70,0.57,-13.0
2,2015-04-29,"April 29, 2015\nFor immediate release Share\nI...",https://www.federalreserve.gov/newsevents/pres...,2015-04-28,0.56,0.56,0.0
3,2015-06-17,"June 17, 2015\nFor immediate release Share\nIn...",https://www.federalreserve.gov/newsevents/pres...,2015-06-16,0.71,0.67,-4.0
4,2015-07-29,"July 29, 2015\nFor immediate release Share\nIn...",https://www.federalreserve.gov/newsevents/pres...,2015-07-28,0.69,0.70,1.0


In [25]:
missing_yields = fomc_yield_df[
    fomc_yield_df["yield_event_close"].isna()
].copy()

print("FOMC announcements without same-day yields:")
display(
    missing_yields[
        ["date", "source_url"]
    ]
)

print("\nYield change summary (basis points):")
print(fomc_yield_df["yield_change_bps"].describe())

print("\nMissing values:")
print(
    fomc_yield_df[
        [
            "yield_previous_close",
            "yield_event_close",
            "yield_change_bps"
        ]
    ].isna().sum()
)

FOMC announcements without same-day yields:


,date,source_url
43,2020-03-15,https://www.federalreserve.gov/newsevents/pres...



Yield change summary (basis points):
count    83.000000
mean     -2.096386
std       7.281139
min     -27.000000
25%      -6.000000
50%      -1.000000
75%       2.000000
max      11.000000
Name: yield_change_bps, dtype: float64

Missing values:
yield_previous_close    1
yield_event_close       1
yield_change_bps        1
dtype: int64


In [26]:
# Preserve the original FOMC announcement date.
fomc_yield_df["yield_observation_date"] = (
    fomc_yield_df["date"]
)

# Identify announcements without same-day yields.
missing_mask = (
    fomc_yield_df["yield_event_close"].isna()
)

# Match missing announcements to the next
# available Treasury trading date.
missing_events = fomc_yield_df.loc[
    missing_mask, ["date"]
].sort_values("date")

next_trading_yields = pd.merge_asof(
    missing_events,
    treasury_aligned[
        [
            "date",
            "previous_trading_date",
            "yield_previous_close",
            "yield_event_close",
            "yield_change_bps"
        ]
    ].sort_values("date"),
    on="date",
    direction="forward",
    tolerance=pd.Timedelta(days=4)
)

# Get the actual observation date.
next_dates = pd.merge_asof(
    missing_events,
    treasury_aligned[
        ["date"]
    ].rename(
        columns={"date": "yield_observation_date"}
    ).sort_values("yield_observation_date"),
    left_on="date",
    right_on="yield_observation_date",
    direction="forward",
    tolerance=pd.Timedelta(days=4)
)

next_trading_yields["yield_observation_date"] = (
    next_dates["yield_observation_date"]
)

# Populate missing yield information.
for column in [
    "previous_trading_date",
    "yield_previous_close",
    "yield_event_close",
    "yield_change_bps",
    "yield_observation_date"
]:
    fomc_yield_df.loc[
        missing_mask, column
    ] = next_trading_yields[column].to_numpy()

# Flag whether the yield observation is on
# the actual announcement date.
fomc_yield_df["same_day_yield"] = (
    fomc_yield_df["date"]
    == fomc_yield_df["yield_observation_date"]
)

print("Same-day matches:",
      fomc_yield_df["same_day_yield"].sum())

print("Next-trading-day matches:",
      (~fomc_yield_df["same_day_yield"]).sum())

display(
    fomc_yield_df.loc[
        fomc_yield_df["date"]
        == pd.Timestamp("2020-03-15"),
        [
            "date",
            "previous_trading_date",
            "yield_observation_date",
            "yield_previous_close",
            "yield_event_close",
            "yield_change_bps",
            "same_day_yield"
        ]
    ]
)

Same-day matches: 83
Next-trading-day matches: 1


,date,previous_trading_date,yield_observation_date,yield_previous_close,yield_event_close,yield_change_bps,same_day_yield
43,2020-03-15,2020-03-13,2020-03-16,0.49,0.36,-13.0,False


In [27]:
# Verify all 84 announcements have yields.
assert len(fomc_yield_df) == 84

assert fomc_yield_df[
    "yield_change_bps"
].notna().all()

assert fomc_yield_df["date"].is_unique

# Verify the weekend event was aligned correctly.
weekend_event = fomc_yield_df.loc[
    fomc_yield_df["date"]
    == pd.Timestamp("2020-03-15")
].iloc[0]

assert weekend_event["previous_trading_date"] == (
    pd.Timestamp("2020-03-13")
)

assert weekend_event["yield_observation_date"] == (
    pd.Timestamp("2020-03-16")
)

print("Final alignment validation successful!")
print("Total announcements:", len(fomc_yield_df))

print("\nYield changes (basis points):")
print(fomc_yield_df["yield_change_bps"].describe())

Final alignment validation successful!
Total announcements: 84

Yield changes (basis points):
count    84.000000
mean     -2.226190
std       7.334275
min     -27.000000
25%      -6.000000
50%      -1.500000
75%       2.000000
max      11.000000
Name: yield_change_bps, dtype: float64


In [28]:
aligned_output_path = (
    PROCESSED_DIR
    / "fomc_treasury_aligned_2015_2024.csv"
)

fomc_yield_df.to_csv(
    aligned_output_path,
    index=False,
    encoding="utf-8"
)

print("Aligned dataset saved successfully!")
print("File:", aligned_output_path)
print("Shape:", fomc_yield_df.shape)

Aligned dataset saved successfully!
File: /git_root/class_project/msml610/Fall2026/projects/UmdTask657_MSML610_Fall2026_WB_Weave_Project1/data/processed/fomc_treasury_aligned_2015_2024.csv
Shape: (84, 9)


## Problem Definition and Baseline Sentence Classification

The objective is to classify Federal Open Market Committee
(FOMC) sentences as hawkish, dovish, or neutral.

A TF-IDF vectorizer and logistic regression classifier
serve as the baseline model.

The model is trained on the labeled training dataset and
evaluated on the held-out test dataset using macro-F1.

Statement-level monetary policy tone is defined as the
proportion of hawkish sentences minus the proportion of
dovish sentences.

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline

# Load the previously cleaned and cached datasets.
train_data = pd.read_csv(PROCESSED_DIR / "train_clean.csv")
test_data = pd.read_csv(PROCESSED_DIR / "test_clean.csv")

# Separate sentences and their labels.
X_train = train_data["sentence"]
y_train = train_data["label"]

X_test = test_data["sentence"]
y_test = test_data["label"]

# Create a reproducible TF-IDF + logistic regression baseline.
baseline_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            lowercase=True,
            max_features=10000,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
    ),
])

# Fit only on training data.
baseline_model.fit(X_train, y_train)

# Predict labels for the held-out test set.
y_pred = baseline_model.predict(X_test)

# Evaluate the baseline.
macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0,
)

accuracy = accuracy_score(y_test, y_pred)

print(f"Baseline macro-F1: {macro_f1:.4f}")
print(f"Baseline accuracy: {accuracy:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0,
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Baseline macro-F1: 0.5180
Baseline accuracy: 0.6008

Classification report:
              precision    recall  f1-score   support

           0       0.64      0.33      0.44       135
           1       0.65      0.30      0.41       114
           2       0.59      0.89      0.71       247

    accuracy                           0.60       496
   macro avg       0.63      0.51      0.52       496
weighted avg       0.62      0.60      0.56       496


Confusion matrix:
[[ 45   4  86]
 [ 11  34  69]
 [ 14  14 219]]


In [30]:
def calculate_statement_tone(
    predicted_labels,
    hawkish_label,
    dovish_label,
):
    """Calculate hawkish share minus dovish share."""
    labels = list(predicted_labels)

    if not labels:
        raise ValueError("A statement must contain sentences.")

    hawkish_share = sum(
        label == hawkish_label for label in labels
    ) / len(labels)

    dovish_share = sum(
        label == dovish_label for label in labels
    ) / len(labels)

    return hawkish_share - dovish_share

In [31]:
# Official label mapping for the FOMC dataset.
LABEL_MAP = {
    0: "Dovish",
    1: "Hawkish",
    2: "Neutral",
}

HAWKISH_LABEL = 1
DOVISH_LABEL = 0
NEUTRAL_LABEL = 2

# Display baseline results with descriptive class names.
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2],
        target_names=["Dovish", "Hawkish", "Neutral"],
        zero_division=0,
    )
)

# Verify the statement-level tone calculation.
example_labels = [1] * 8 + [0] * 5 + [2] * 7

example_tone = calculate_statement_tone(
    example_labels,
    hawkish_label=HAWKISH_LABEL,
    dovish_label=DOVISH_LABEL,
)

assert abs(example_tone - 0.15) < 1e-10
print(f"Example statement tone: {example_tone:+.2f}")

              precision    recall  f1-score   support

      Dovish       0.64      0.33      0.44       135
     Hawkish       0.65      0.30      0.41       114
     Neutral       0.59      0.89      0.71       247

    accuracy                           0.60       496
   macro avg       0.63      0.51      0.52       496
weighted avg       0.62      0.60      0.56       496

Example statement tone: +0.15
